# Install

In [ ]:
!pip3 install pshmodule

In [ ]:
!pip3 install pickle5

In [ ]:
!pip3 install pandas==1.5.0

In [ ]:
!pip3 install swifter

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Data Load

In [ ]:
import sys
sys.path.append('/content/drive/MyDrive/Advertising_By_Personality/src/business_model/keyword_extraction')
print(sys.path)

['/content', '/env/python', '/usr/lib/python310.zip', '/usr/lib/python3.10', '/usr/lib/python3.10/lib-dynload', '', '/usr/local/lib/python3.10/dist-packages', '/usr/lib/python3/dist-packages', '/usr/local/lib/python3.10/dist-packages/IPython/extensions', '/root/.ipython', '/content/drive/MyDrive/Advertising_By_Personality/src/business_model/keyword_extraction']


In [ ]:
import re
import json
import config as cfg
import swifter
import pandas as pd
from tqdm import tqdm
from pshmodule.utils import filemanager as fm

In [ ]:
import json
with open("drive/MyDrive/Advertising_By_Personality/data/gpt/ext_gpt_nt2.json", "r", encoding="utf-8") as f:
    data = json.load(f)

In [ ]:
df_nt = pd.DataFrame(data)

In [ ]:
df_nt.shape

(1000, 13)

In [ ]:
data = []
with open("drive/MyDrive/Advertising_By_Personality/data/gpt/ext_gpt_nf2.json", 'r', encoding='utf-8') as f:
    for line in f:
        data.append(json.loads(line.rstrip('\n|\r')))
df_nf = pd.DataFrame(data)

In [ ]:
df_nf.shape

(1000, 13)

In [ ]:
df = pd.concat([df_nt, df_nf], axis=0)

In [ ]:
df.reset_index(inplace=True, drop=True)

In [ ]:
df.shape

(2000, 13)

In [ ]:
df.label.head()

0    이 혜택💰 놓치지 않을 거예요\\[계절밥상] 디너/주말 1O,OOO원, 런치 5,O...
1    무조건 당첨인데 참여 외않해?👀\\5OOP만 있으면 무조건 당첨되는 갓-이벤트✨\2...
2    아직 손도 안 댄 쿠폰이 있어요!\\미사용 쿠폰 만료까지 D-7🚨 쿠폰함으로 확인하...
3    주말엔 사라질 그 할인쿠폰!\\[OOOO 1만원 할인쿠폰] 의 유효기간은 이번 주 ...
4    OO 바캉스 위크 특가⚡누리세오!\\아참, 득템프 1회 더 찍으면 5,OOO원 할인...
Name: label, dtype: object

# Data Check

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   marketing_entity    2000 non-null   object
 1   marketing_target    2000 non-null   object
 2   benefit_conditions  2000 non-null   object
 3   benefits            2000 non-null   object
 4   discount_figure     2000 non-null   object
 5   promotional_items   2000 non-null   object
 6   promotional_place   2000 non-null   object
 7   event_period        2000 non-null   object
 8   dow_information     2000 non-null   object
 9   season_information  2000 non-null   object
 10  solicitation_point  2000 non-null   object
 11  type                2000 non-null   object
 12  label               2000 non-null   object
dtypes: object(13)
memory usage: 203.2+ KB


In [ ]:
df.solicitation_point.value_counts().iloc[201:250]

서울의 아름다운 이 도시.                      1
막차타러                                1
온라인몰                                1
대구 볼로냐 국제 일러전, 부산 신카이 마코토전, 부산타워    1
CJ ONE 금융타운                         1
소구점: 신세계면세점                         1
한가위 풍성한 선물세트                        1
맛잘알들의 성지 뚜레쥬르                       1
슈가플래닛                               1
내가 쓴 포인트 되찾기                        1
제주 [빛의 벙커: 클림트]                     1
이태원                                 1
하이메아욘 전시를 보러 오실 것으로 예상              1
우리집                                 1
상하이                                 1
소구점 :                               1
[TAG1]님을 위한 것                       1
광주 곳곳                               1
CJ푸드월드 코엑스몰점.                       1
영화 데이트                              1
존버거맨 전시                             1
OCN스릴러하우스                           1
신라                                  1
갬동                                  1
인천공항 내 올영                           1
집밥 맛집                               1
앱           

# Processing

각각 에러 데이터 제거 2번 작업

In [ ]:
df.at[1988, 'marketing_entity'] = '없음'
df.at[1988, 'benefit_conditions'] = '없음'
df.at[1988, 'discount_figure'] = '없음'
df.at[1988, 'promotional_place'] = '없음'
df.at[1988, 'dow_information'] = '없음'
df.at[1988, 'solicitation_point'] = '없음'

In [ ]:
df['marketing_entity'] = df.marketing_entity.apply(lambda x: x.replace('없음, ', '').replace('없음, TAG1', 'TAG1').replace('(불명)', '없음').replace('<br>', '').replace('없음 (스팸 판매 업체)', '스팸 판매 업체').replace(' (정보 제공 없음)', '').replace('/TAG1', '').replace('/', ', ').replace("'", ""))
df['marketing_target'] = df.marketing_target.apply(lambda x: x.replace('있음 (또 오실 거죠?)', '없음'). replace('/', ', ').replace('타겟 :', '없음').replace(' or ', ','))
df['benefit_conditions'] = df.benefit_conditions.apply(lambda x: x.replace('<br>', '').replace('혜택 지급 조건 :', '없음').replace(' or ', ', '))
df['benefits'] = df.benefits.apply(lambda x: x.replace('<br>', '').replace(' (상세 내용 미기재)', '').replace(' (할인 정보 없음)', ''))
df['discount_figure'] = df.discount_figure.apply(lambda x: x.replace('있으나 문장에서 언급되지 않음', '없음').replace('할인 정보 없음', '없음'))
df['promotional_items'] = df.promotional_items.apply(lambda x: x.replace('<br>', ''))
df['promotional_place'] = df.promotional_place.apply(lambda x: x.replace('(미기재)', ''))
df['event_period'] = df.event_period.apply(lambda x: x.replace('(안내 X)', '').replace('없음 (선착순 할인 이벤트 시작 시간은 11시)', '선착순 할인 이벤트 시작 시간은 11시').replace('이벤트 기간 :', '없음'))
df['season_information'] = df.season_information.apply(lambda x: x.replace('<br>', '').replace('시즌 정보 :', '없음'))
df['solicitation_point'] = df.solicitation_point.apply(lambda x: x.replace('(소구점 정보 없음)', '').replace('소구점 :', '없음'))

In [ ]:
df['marketing_entity'] = df.marketing_entity.apply(lambda x: x.replace('마케팅 주체 : ', '').replace('마케팅 주체: ', ''))
df['marketing_target'] = df.marketing_target.apply(lambda x: x.replace('타겟 : ', '').replace('타겟: ', ''))
df['benefit_conditions'] = df.benefit_conditions.apply(lambda x: x.replace('혜택 지급 조건 : ', '').replace('혜택 지급 조건: ', ''))
df['benefits'] = df.benefits.apply(lambda x: x.replace('혜택 : ', '').replace('혜택: ', ''))
df['discount_figure'] = df.discount_figure.apply(lambda x: x.replace('할인 수치 : ', '').replace('할인 수치: ', ''))
df['promotional_items'] = df.promotional_items.apply(lambda x: x.replace('프로모션 품목 : ', '').replace('프로모션 품목: ', ''))
df['promotional_place'] = df.promotional_place.apply(lambda x: x.replace('프로모션 장소 : ', '').replace('프로모션 장소: ', ''))
df['event_period'] = df.event_period.apply(lambda x: x.replace('이벤트 기간 : ', '').replace('이벤트 기간: ', ''))
df['dow_information'] = df.dow_information.apply(lambda x: x.replace('요일 정보 : ', '').replace('요일 정보: ', ''))
df['season_information'] = df.season_information.apply(lambda x: x.replace('시즌 정보 : ', '').replace('시즌 정보: ', ''))
df['solicitation_point'] = df.solicitation_point.apply(lambda x: x.replace('소구점 : ', '').replace('소구점: ', ''))

In [ ]:
df['marketing_entity'] = df.marketing_entity.apply(lambda x: x.replace('[', '').replace(']', '').replace('+', ',').replace('O', '0').replace(' / ',', ').rstrip(',').rstrip('，').rstrip('.').strip())
df['marketing_target'] = df.marketing_target.apply(lambda x: x.replace('[', '').replace(']', '').replace('+', ',').replace('O', '0').replace(' / ',', ').rstrip(',').rstrip('，').rstrip('.').strip())
df['benefit_conditions'] = df.benefit_conditions.apply(lambda x: x.replace('[', '').replace(']', '').replace('+', ',').replace('O', '0').replace(' / ',', ').rstrip(',').rstrip('，').rstrip('.').strip())
df['benefits'] = df.benefits.apply(lambda x: x.replace('[', '').replace(']', '').replace('+', ',').replace('O', '0').replace(' / ',', ').rstrip(',').rstrip('，').rstrip('.').strip())
df['discount_figure'] = df.discount_figure.apply(lambda x: x.replace('[', '').replace(']', '').replace('+', ',').replace('O', '0').replace(' / ',', ').rstrip(',').rstrip('，').rstrip('.').strip())
df['promotional_items'] = df.promotional_items.apply(lambda x: x.replace('[', '').replace(']', '').replace('+', ',').replace('O', '0').replace(' / ',', ').rstrip(',').rstrip('，').rstrip('.').strip())
df['promotional_place'] = df.promotional_place.apply(lambda x: x.replace('[', '').replace(']', '').replace('+', ',').replace('O', '0').replace(' / ',', ').rstrip(',').rstrip('，').rstrip('.').strip())
df['event_period'] = df.event_period.apply(lambda x: x.replace('[', '').replace(']', '').replace('+', ',').replace('O', '0').replace(' / ',', ').rstrip(',').rstrip('，').rstrip('.').strip())
df['dow_information'] = df.dow_information.apply(lambda x: x.replace('[', '').replace(']', '').replace('+', ',').replace('O', '0').replace(' / ',', ').rstrip(',').rstrip('，').rstrip('.').strip())
df['season_information'] = df.season_information.apply(lambda x: x.replace('[', '').replace(']', '').replace('+', ',').replace('O', '0').replace(' / ',', ').rstrip(',').rstrip('，').rstrip('.').strip())
df['solicitation_point'] = df.solicitation_point.apply(lambda x: x.replace('[', '').replace(']', '').replace('+', ',').replace('O', '0').replace(' / ',', ').rstrip(',').rstrip('，').rstrip('.').strip())
df['label'] = df.label.apply(lambda x: x.replace('O', '0').strip())

In [ ]:
df.fillna('없음', inplace=True)

### Save

In [ ]:
temp_dict = [{"marketing_entity": row['marketing_entity'], "marketing_target": row['marketing_target'], "benefit_conditions": row['benefit_conditions'], "benefits": row['benefits'], "discount_figure": row['discount_figure'], "promotional_items": row['promotional_items'], "promotional_place": row['promotional_place'], "event_period": row['event_period'], "dow_information": row['dow_information'], "season_information": row['season_information'], "solicitation_point": row['solicitation_point'], "type": row['type'], "label": row['label']} for _, row in df.iterrows()]
with open("drive/MyDrive/Advertising_By_Personality/data/gpt/train_ntnf2.json", 'w', encoding='utf-8') as f:
    for line in temp_dict:
        json_record = json.dumps(line, ensure_ascii=False)
        f.write(json_record + '\n')

# Unicode

file load

In [ ]:
import json
import pandas as pd

In [ ]:
data = []
with open("drive/MyDrive/Advertising_By_Personality/data/gpt/ext_gpt_mbti.json", 'r', encoding='utf-8') as f:
    for line in f:
        data.append(json.loads(line.rstrip('\n|\r')))
df = pd.DataFrame(data)

unicode

In [ ]:
import swifter
from pshmodule.processing import processing as p

In [ ]:
nlp = p.Nlp()

In [ ]:
df.columns

Index(['NT', 'NF', 'marketing_entity', 'marketing_target',
       'benefit_conditions', 'benefits', 'discount_figure',
       'promotional_items', 'promotional_place', 'event_period',
       'dow_information', 'season_information', 'solicitation_point', 'type',
       'label'],
      dtype='object')

In [ ]:
df['NT'] = df.NT.swifter.apply(lambda x: nlp.convert_to_other_unicode(x))
df['NF'] = df.NF.swifter.apply(lambda x: nlp.convert_to_other_unicode(x))
df['marketing_entity'] = df.marketing_entity.swifter.apply(lambda x: nlp.convert_to_other_unicode(x))
df['marketing_target'] = df.marketing_target.swifter.apply(lambda x: nlp.convert_to_other_unicode(x))
df['benefit_conditions'] = df.benefit_conditions.swifter.apply(lambda x: nlp.convert_to_other_unicode(x))
df['benefits'] = df.benefits.swifter.apply(lambda x: nlp.convert_to_other_unicode(x))
df['discount_figure'] = df.discount_figure.swifter.apply(lambda x: nlp.convert_to_other_unicode(x))
df['promotional_items'] = df.promotional_items.swifter.apply(lambda x: nlp.convert_to_other_unicode(x))
df['promotional_place'] = df.promotional_place.swifter.apply(lambda x: nlp.convert_to_other_unicode(x))
df['event_period'] = df.event_period.swifter.apply(lambda x: nlp.convert_to_other_unicode(x))
df['dow_information'] = df.dow_information.swifter.apply(lambda x: nlp.convert_to_other_unicode(x))
df['season_information'] = df.season_information.swifter.apply(lambda x: nlp.convert_to_other_unicode(x))
df['solicitation_point'] = df.solicitation_point.swifter.apply(lambda x: nlp.convert_to_other_unicode(x))
df['label'] = df.label.swifter.apply(lambda x: nlp.convert_to_other_unicode(x))

Pandas Apply:   0%|          | 0/2000 [00:00<?, ?it/s]

Pandas Apply:   0%|          | 0/2000 [00:00<?, ?it/s]

Pandas Apply:   0%|          | 0/2000 [00:00<?, ?it/s]

Pandas Apply:   0%|          | 0/2000 [00:00<?, ?it/s]

Pandas Apply:   0%|          | 0/2000 [00:00<?, ?it/s]

...

Pandas Apply:   0%|          | 0/2000 [00:00<?, ?it/s]

Pandas Apply:   0%|          | 0/2000 [00:00<?, ?it/s]

Pandas Apply:   0%|          | 0/2000 [00:00<?, ?it/s]

Pandas Apply:   0%|          | 0/2000 [00:00<?, ?it/s]

Pandas Apply:   0%|          | 0/2000 [00:00<?, ?it/s]

In [ ]:
df.head()

,NT,NF,marketing_entity,marketing_target,benefit_conditions,benefits,discount_figure,promotional_items,promotional_place,event_period,dow_information,season_information,solicitation_point,type,label
0,"이 혜택\ud83d\udcb0 놓치지 않을 거예요, [계절밥상] 디너/주말 10,0...",없음,계절밥상,없음,없음,"디너/주말 10,000원, 런치 5,000원 할인","10,000원 디너/주말, 5,000원 런치",계절밥상,없음,9월 30일까지,없음,없음,없음,NT,"이 혜택\ud83d\udcb0 놓치지 않을 거예요\\[계절밥상] 디너/주말 10,0..."
1,"무조건 당첨인데 참여 외않해?\ud83d\udc40, 500P만 있으면 무조건 당첨...",없음,없음,없음,500P,"200만원 상당 여행권, 빕스 쿠폰",없음,없음,없음,없음,없음,없음,없음,NT,무조건 당첨인데 참여 외않해?\ud83d\udc40\\500P만 있으면 무조건 당첨...
2,"아직 손도 안 댄 쿠폰이 있어요!, 미사용 쿠폰 만료까지 D-7\ud83d\udea...",없음,없음,없음,미사용 쿠폰 만료까지 확인,없음,없음,없음,없음,D-7,없음,없음,없음,NT,아직 손도 안 댄 쿠폰이 있어요!\\미사용 쿠폰 만료까지 D-7\ud83d\udea...
3,"주말엔 사라질 그 할인쿠폰!, [0000 1만원 할인쿠폰], 유효기간은 이번 주 일...",없음,없음,없음,할인 쿠폰 사용,1만원 할인,10000원,통돼지구이,없음,이번 주 일요일까지,일요일,없음,없음,NT,주말엔 사라질 그 할인쿠폰!\\[0000 1만원 할인쿠폰] 의 유효기간은 이번 주 ...
4,"득템프 1회 더 찍으면 5,000원 할인 쿠폰도 드린답니다","아참, 누리세오!",누리세오,없음,득템프 1회 더 찍기,"5,000원 할인 쿠폰","5,000원",바캉스 위크,없음,없음,없음,없음,없음,NT,"00 바캉스 위크 특가⚡누리세오!\\아참, 득템프 1회 더 찍으면 5,000원 할인..."


# Save

1차

In [ ]:
data = []
with open("drive/MyDrive/Advertising_By_Personality/data/final_train.json", 'r', encoding='utf-8') as f:
    for line in f:
        data.append(json.loads(line.rstrip('\n|\r')))
df1 = pd.DataFrame(data)

2차

In [ ]:
df = pd.concat([df1, df2], axis=0)

1, 2차 합치기

In [ ]:
temp_dict = [{"NT": row['NT'], "NF": row['NF'], "marketing_entity": row['marketing_entity'], "marketing_target": row['marketing_target'], "benefit_conditions": row['benefit_conditions'], "benefits": row['benefits'], "discount_figure": row['discount_figure'], "promotional_items": row['promotional_items'], "promotional_place": row['promotional_place'], "event_period": row['event_period'], "dow_information": row['dow_information'], "season_information": row['season_information'], "solicitation_point": row['solicitation_point'], "type": row['type'], "label": row['label']} for _, row in df.iterrows()]
with open("drive/MyDrive/Advertising_By_Personality/data/train_final2.json", 'w', encoding='utf-8') as f:
    for line in temp_dict:
        json_record = json.dumps(line, ensure_ascii=False)
        f.write(json_record + '\n')